1. The following data yield the arrival times and service times that each customer will require, for the first 13 customers at a single server system. Upon arrival, a customer either enters service if the server is free or joins the waiting line. When the server completes work on a customer, the next one in line (i.e., the one who has been waiting the longest) enters service.

| Arrival Times: | 12  | 31  | 63  | 95  | 99  | 154 | 198 | 221 | 304 | 346 | 411 | 455 | 537 |
| -------------- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Service Times: | 40  | 32  | 55  | 48  | 18  | 50  | 47  | 18  | 28  | 54  | 40  | 72  | 12  |


In [32]:
arrivals = [12, 31, 63, 95, 99, 154, 198, 221, 304, 346, 411, 455, 537]
services = [40, 32, 55, 48, 18, 50, 47, 18, 28, 54, 40, 72, 12]
N = len(arrivals)

(a) Determine the departure times of these 13 customers

In [33]:
free_time = 0  # 服務檯下一次空閒時間
depa_1a = []
for a, s in zip(arrivals, services):
    free_time = (
        max(free_time, a) + s
    )  # 如果服務檯繁忙，需要等待到空閒，然後加上服務時間
    depa_1a.append(free_time)
depa_1a

[52, 84, 139, 187, 205, 255, 302, 320, 348, 402, 451, 527, 549]

(b) Repeat (a) when there are two servers and a customer can be served by either one.


In [34]:
free_time = [0, 0]  # 2個服務檯下一次空閒時間
depa_1b = []
for a, s in zip(arrivals, services):
    next_customer = (
        0 if free_time[0] <= free_time[1] else 1
    )  # 根據下次空閒時間計算下一個空間的服務檯
    free_time[next_customer] = max(free_time[next_customer], a) + s
    depa_1b.append(free_time[next_customer])
depa_1b

[52, 63, 118, 143, 136, 204, 245, 239, 332, 400, 451, 527, 549]

Repeat (a) under the new assumption that when the server completes a service. The next customer to enter service is the one who has been waiting the least time.


In [35]:
free_time = 0
wait_stack = []  # 記錄已經在等待的顧客
departure = [0] * N
next_customer = 0  # 從第一位顧客開始
coming_customer = next_customer + 1  # 記錄要到來的顧客
while next_customer < N or wait_stack:  # 直到沒有任何顧客會到來，並且等待顧客都服務完
    free_time = max(free_time, arrivals[next_customer]) + services[next_customer]
    departure[next_customer] = free_time

    # 將服務結束前到達的客戶加入等待列隊
    while coming_customer < N and arrivals[coming_customer] <= free_time:
        wait_stack.append(coming_customer)
        coming_customer += 1

    if wait_stack:  # 如果有人等待，則服務最後到達的客戶
        next_customer = wait_stack.pop()
    else:  # 無人等待就直接服務下一個到達的
        next_customer = coming_customer
        coming_customer += 1

print(departure)


[52, 84, 139, 320, 157, 207, 254, 272, 348, 402, 451, 527, 549]


2. Consider a service station where customers arrive and are served in their order of arrival. Let $A_n$, $S_n$, and $D_n$ denote, respectively, the arrival time, the service time, and the departure time of customer $n$. Suppose there is a single server and that the system is initially empty of customers.

(a) With $D_0 = 0$, argue that for $n > 0$
$$D_n - S_n = \max\{A_n, D_{n-1}\}$$



$D_n - S_n = n$ 開始被服務的時間。如果當 $n$ 到達時服務檯空閒，可以立即開始服務，如果有人正在被服務，則需要等待前一位客戶服務完成並離開。

(b) Determine the corresponding recursion formula when there are two servers.

Let $T_n^i=$ 安排完第 $n$ 個客戶後第 $i$ 個服務檯完成服務的時間，$i=1,2$。當有一個服務檯空閒，即可服務下一個客戶。

WLOG 若 $T_{n-1}^1 \le T_{n-1}^2$，則 $T_n^1 = D_n, T_n^2 = T_{n-1}^2$

$$D_n - S_n = \max\{A_n, \min(T_{n-1}^1, T_{n-1}^2)\}$$


(c) Determine the corresponding recursion formula when there are $k$ servers.

Let $T_n^i=$ 安排完第 $n$ 個客戶後第 $i$ 個服務檯完成服務的時間，$i=1,\dots,k$。當有一個服務檯空閒，即可服務下一個客戶。

WLOG 若 $T_{n-1}^j = \min_i\set{T_{n-1}^i}$，則 $T_n^j = D_n, T_n^i = T_{n-1}^i,i\neq j$

$$D_n - S_n = \max\{A_n, \min_i\set{T_{n-1}^i}\}$$

(d) Write a computer program to determine the departure times as a function of the arrival and service times and use it to check your answers in parts (a) and (b) of Exercise 1.

In [36]:
def func(arrivals: list[int], services: list[int], n_servers: int) -> list[int]:
    assert len(arrivals) == len(services)  # 聲明兩個列表長度一致

    free_time = [0] * n_servers  # k 個服務檯下一次空閒時間
    depa = []
    for a, s in zip(arrivals, services):
        # 尋找最早完成服務的服務檯
        min_t_idx = 0
        for i, t in enumerate(free_time):
            if t < free_time[min_t_idx]:
                min_t_idx = i

        free_time[min_t_idx] = max(free_time[min_t_idx], a) + s
        depa.append(free_time[min_t_idx])
    return depa

In [37]:
depa_2c1 = func(arrivals, services, 1)
print(depa_2c1)
depa_2c2 = func(arrivals, services, 2)
print(depa_2c2)
print(depa_2c1 == depa_1a and depa_2c2 == depa_1b)

[52, 84, 139, 187, 205, 255, 302, 320, 348, 402, 451, 527, 549]
[52, 63, 118, 143, 136, 204, 245, 239, 332, 400, 451, 527, 549]
True


21. Explain how to make use of the relationship

    $$p_{i+1} = \frac{\lambda}{i+1} p_i$$

    to compute efficiently the Poisson probabilities.

In [40]:
# 快速計算大量 p 值，而不用重複計算階乘。且指數運算相比乘除，運算速度非常慢。

from math import exp

lam = 1.5
p = [0] * 30
p[0] = exp(-lam)
for i in range(1, len(p)):
    p[i] = lam * p[i - 1] / i
p

[0.22313016014842982,
 0.33469524022264474,
 0.25102143016698353,
 0.12551071508349176,
 0.04706651815630941,
 0.014119955446892823,
 0.0035299888617232058,
 0.0007564261846549727,
 0.00014182990962280736,
 2.3638318270467892e-05,
 3.545747740570184e-06,
 4.835110555322979e-07,
 6.043888194153724e-08,
 6.973717147100451e-09,
 7.471839800464768e-10,
 7.471839800464768e-11,
 7.00484981293572e-12,
 6.180749834943283e-13,
 5.150624862452736e-14,
 4.066282786146896e-15,
 3.049712089610172e-16,
 2.1783657782929802e-17,
 1.4852493942906682e-18,
 9.68640909320001e-20,
 6.054005683250006e-21,
 3.6324034099500037e-22,
 2.095617351894233e-23,
 1.1642318621634627e-24,
 6.236956404447121e-26,
 3.226011933334718e-27]

29. Persons A, B, and C are waiting at a bank having two tellers when it opens in the morning. Persons A and B each go to a teller and C waits in line. If the time it takes to serve a customer is an exponential random variable with parameter $\lambda$, what is the probability that C is the last to leave the bank? [Hint: No computations are necessary.]

WLOG, assume A finished before B at time $t$, by memoryless prop
$$
\begin{aligned}
&P(B > t + s | B > t) = P(B - t > s | B > t) = P(B > s) \\ 
\implies& B-t | B> t \sim \exp(\lambda)\overset{d}{=} C \\
\implies& P(C > B-t|>t)=1/2
\end{aligned}
$$



25. The bus will arrive at a time that is uniformly distributed between 8 and 8:30 A.M. If we arrive at 8 A.M., what is the probability that we will wait between 5 and 15 minutes?

$$
T \sim U(0, 30), P(5\le T \le 15 ) = \frac{10}{30} = \frac{1}{3}
$$

32. For a Poisson process with rate $\lambda$, find $P\set{N(s) = k | N(t) = n}$ when $s < t$.


在總數已知的情況下樣本均勻落在兩塊區域中，i.e. $N(s) | N(t)=n \sim Bin(n, \frac{s}{t})$

$$P\set{N(s) = k | N(t) = n}=\binom nk (\frac st)^k (1-\frac st)^{n-k}$$

or

$$
N(s) \sim Poi(\lambda s)\perp N(t) - N(s) \sim Poi(\lambda(t-s))
$$

$$
P\set{N(s) = k | N(t) = n} = \frac{\frac{\exp(-\lambda s)(-\lambda s)^k}{k!}\times \frac{\exp(-\lambda (t-s))(-\lambda (t-s))^{n-k}}{(n-k)!}}{\frac{\exp(-\lambda t)(-\lambda t)^n}{n!}}=\frac{n!}{k!(n-k)!}(\frac st)^k (1-\frac st)^{n-k}
$$

33. Repeat Exercise 32 for s > t.

$$N(s) \sim Poi(\lambda s)\perp N(s) - N(t) \sim Poi(\lambda(s-t))$$

$$
\begin{aligned}
&P\set{N(s) = k | N(t) = n} \\ 
=&P\set{N(s) - N(t) = k - n | N(t) = n} \\ 
=& P\set{N(s) - N(t) = k - n} \\
=& \frac{e^{-\lambda(s-t)}(\lambda(s-t))^{k-n}}{(k-n)!}
\end{aligned}
$$